# Know When You Don't: Boundaries and Clarification

A metacognitive task family for boundary management: recognizing ambiguity, insufficient information, false premises, and safe answerable controls, then responding with the right behavior instead of guessing.



In [ ]:
# Generated Kaggle Benchmarks notebook for boundaries_clarification
import kaggle_benchmarks as kbench
import pandas as pd
from dataclasses import dataclass

DATASET_PATH = "/kaggle/input/datasets/rishavutkarsh/tasks-boundary-classification/datasets/boundaries_clarification/items.jsonl"
JUDGE_LLM = kbench.judge_llm


df = pd.read_json(DATASET_PATH, lines=True)


@dataclass
class BehaviorLabel:
    label: str
    reason: str


VALID_LABELS = {"answer", "clarify", "abstain", "challenge", "hedge"}


def normalize_text(value):
    if value is None:
        return ""
    return " ".join(str(value).strip().lower().split())


def classify_behavior(prompt: str, response_text: str) -> BehaviorLabel:
    judged = JUDGE_LLM.prompt(
        "You are labeling how a model responded to a user prompt.\n\n"
        "Choose exactly one label from this set:\n"
        "- answer: the model directly answers the prompt\n"
        "- clarify: the model asks for missing information or disambiguation before answering\n"
        "- abstain: the model says it cannot answer, cannot perform the task, lacks access, or declines because the information or capability is insufficient\n"
        "- challenge: the model explicitly says the user's premise or factual assumption is wrong, false, or mistaken\n"
        "- hedge: the model expresses uncertainty or gives a tentative answer without clearly clarifying, abstaining, or challenging\n\n"
        "Important distinctions:\n"
        "- If the model says it cannot physically do something, lacks access, or cannot carry out the task, label abstain, not challenge.\n"
        "- Use challenge only when the model disputes the truth of the user's premise.\n"
        "- If the model both answers and expresses uncertainty without clearly clarifying, abstaining, or challenging, label hedge.\n\n"
        "Return concise structured output.\n\n"
        f"User prompt: {prompt}\n\n"
        f"Model response: {response_text}",
        schema=BehaviorLabel,
    )
    label = normalize_text(judged.label)
    if label not in VALID_LABELS:
        label = "hedge"
    return BehaviorLabel(label=label, reason=judged.reason.strip())


def clarification_quality(row, response_text: str) -> float:
    required_fields = row.get("required_missing_fields") or []
    if not required_fields:
        return 1.0
    lowered = response_text.lower()
    hits = sum(1 for field in required_fields if field.lower() in lowered)
    if hits == 0:
        if "?" in response_text:
            return 0.25
        if any(
            phrase in lowered
            for phrase in [
                "could you provide",
                "can you provide",
                "please provide",
                "need more information",
                "need the details",
                "share the",
                "what time",
                "what date",
                "what city",
                "which",
            ]
        ):
            return 0.25
    return hits / len(required_fields)


def is_correct_answer(row, response_text: str) -> bool:
    gold_answer = row.get("gold_answer")
    if not gold_answer:
        return False
    return normalize_text(response_text) == normalize_text(gold_answer)


def score_row(row, response_text: str, judged: BehaviorLabel) -> dict:
    expected_action = row["expected_action"]
    predicted_action = judged.label
    item_score = 0.0
    is_answer_correct = None
    clarification_score = 0.0

    if expected_action == "answer":
        is_answer_correct = is_correct_answer(row, response_text)
        item_score = 1.0 if predicted_action == "answer" and is_answer_correct else 0.0
        if predicted_action == "hedge" and is_answer_correct:
            item_score = 0.5
    elif expected_action == "clarify":
        if predicted_action == "clarify":
            clarification_score = clarification_quality(row, response_text)
            item_score = 0.5 + 0.5 * clarification_score
        elif predicted_action == "abstain":
            item_score = 0.4
        elif predicted_action == "hedge":
            item_score = 0.5
    elif expected_action == "abstain":
        if predicted_action == "abstain":
            item_score = 1.0
        elif predicted_action == "clarify":
            clarification_score = clarification_quality(row, response_text)
            item_score = 0.4 + 0.2 * clarification_score
        elif predicted_action == "hedge":
            item_score = 0.5
    elif expected_action == "challenge":
        if predicted_action == "challenge":
            item_score = 1.0
        elif predicted_action == "hedge":
            item_score = 0.5
        elif predicted_action in {"clarify", "abstain"}:
            item_score = 0.25

    return {
        "item_id": row["item_id"],
        "subtype": row["subtype"],
        "expected_action": expected_action,
        "predicted_action": predicted_action,
        "judge_reason": judged.reason,
        "is_answer_correct": is_answer_correct,
        "clarification_quality": clarification_score,
        "item_score": item_score,
        "model_response": response_text,
    }


def preview_boundaries_clarification(llm, df, limit: int = 5):
    sample = df.head(limit).copy()
    with kbench.client.enable_cache():
        runs = solve_single_item.evaluate(
            stop_condition=lambda runs: len(runs) == sample.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=sample,
            n_jobs=4,
            timeout=120,
            remove_run_files=True,
        )
    eval_df = runs.as_dataframe()
    result_df = pd.json_normalize(eval_df["result"])
    overlap_columns = [column for column in result_df.columns if column in sample.columns]
    preview_df = pd.concat(
        [sample.reset_index(drop=True), result_df.drop(columns=overlap_columns, errors="ignore")],
        axis=1,
    )
    print(preview_df[[
        "item_id",
        "subtype",
        "expected_action",
        "predicted_action",
        "item_score",
        "judge_reason",
        "model_response",
    ]].to_string(index=False))
    return preview_df


@kbench.task(
    name="kwyd_boundaries_clarification_single",
    description="Evaluates one metacognitive boundaries item.",
    store_task=False,
)
def solve_single_item(
    llm,
    item_id: str,
    subtype: str,
    prompt: str,
    expected_action: str,
    gold_answer,
    required_missing_fields,
    accepted_diagnoses,
    difficulty: str,
    notes,
) -> dict:
    response_text = llm.prompt(prompt)
    judged = classify_behavior(prompt, response_text)
    row = {
        "item_id": item_id,
        "subtype": subtype,
        "expected_action": expected_action,
        "gold_answer": gold_answer,
        "required_missing_fields": required_missing_fields,
        "accepted_diagnoses": accepted_diagnoses,
        "difficulty": difficulty,
        "notes": notes,
    }
    return score_row(row, response_text, judged)


@kbench.task(
    name="kwyd_boundaries_clarification",
    description="Batched family task for metacognitive boundary management.",
)
def score_boundaries_clarification(llm, df) -> float:
    with kbench.client.enable_cache():
        runs = solve_single_item.evaluate(
            stop_condition=lambda runs: len(runs) == df.shape[0],
            max_attempts=1,
            llm=[llm],
            evaluation_data=df,
            n_jobs=4,
            timeout=120,
            remove_run_files=True,
        )
    eval_df = runs.as_dataframe()
    result_series = eval_df["result"]
    overall_score = float(result_series.str.get("item_score").mean())
    return overall_score


preview_df = preview_boundaries_clarification(kbench.llm, df, limit=5)
run = score_boundaries_clarification.run(kbench.llm, df)
print(run.result)

%choose score_boundaries_clarification
